<a href="https://colab.research.google.com/github/ShreyaMathur19/Clustered-Diagonal-Segment-Format-CDSF-/blob/main/CSR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CSR (Compressed Sparse Row)**

CSR stores non-zero elements row-wise using values, column indices, and row pointers. We performed SpMV (y = A·x) using CSR and measured execution time and peak memory usage to evaluate its efficiency for row-based computations.

In [ ]:
from scipy.io import mmread
import numpy as np
from scipy.sparse import coo_matrix
from time import perf_counter
import numba as nb

In [ ]:
# Load the matrix from an .mtx file
A = mmread("wang3.mtx")
A = coo_matrix(A)

In [ ]:
#Converting from COO to CSR
def _coo_to_csr(A_csr):
    return A_coo.tocsr()

In [ ]:
#CSR SPMV AND SPMV TIME  CONSUMPTION

def time_spmv_csr(A_csr, reps=20, seed=42):
    n = A_csr.shape[1]
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n).astype(np.float64)

    # warm-up
    _ = A_csr @ x

    t0 = perf_counter()
    for _ in range(reps):
        y = A_csr @ x
    t_ms = (perf_counter() - t0) / reps * 1000.0  # ms per call

    print(f"y = A_csr @ x  : {t_ms:.2f} ms per call")

    return t_ms

In [ ]:
#Time and peak memory while converting from COO to CSR

def time_and_peak_mem_coo_to_csr(A_coo, reps=5, interval=0.001):
    """
    Measures average time (ms) and peak memory (MB) for A_coo.todia().
    Uses memory_profiler to measure peak RSS per run, then reports the max peak.
    """
    times_ms = []
    peak_deltas_mb = []
    A_csr = None

    # One warm-up (not timed) to avoid cold-start noise
    _ = A_coo.tocsr()

    for _ in range(reps):
        t0 = perf_counter()
        mem_trace, A_csr = memory_usage(
            (_coo_to_csr, (A_coo,), {}), retval=True, interval=interval
        )
        elapsed_ms = (perf_counter() - t0) * 1000.0
        # memory_usage returns a list of RSS (MB); take peak - start as delta for the run
        peak_delta_mb = max(mem_trace) - mem_trace[0]

        times_ms.append(elapsed_ms)
        peak_deltas_mb.append(peak_delta_mb)

    avg_ms = float(np.mean(times_ms))
    peak_mb = float(max(peak_deltas_mb))  # the worst (highest) peak across runs

    print(f"COO->CSR: {avg_ms:.2f} ms per call (avg over {reps})")
    print(f"Peak memory during conversion: {peak_mb:.2f} MB")
    return A_csr, avg_ms, peak_mb